# Lyric video - drawings ko animate karo (image-to-video)

**Kaise chalana hai**
1. Upar menu: **Runtime > Change runtime type > T4 GPU > Save**
2. Phir: **Runtime > Run all**
3. Drive ki permission maange to **Allow** karo.
4. Har scene mein kuch minute lagenge. Clips Drive ke `lyric_video_clips` folder mein save hongi.

Agar beech mein ruk jaye to dobara **Run all** dabao - jo clips ban chuki hain wo dobara nahi banengi.
Error aaye to uska screenshot bhej do.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q -U diffusers transformers accelerate sentencepiece imageio imageio-ffmpeg

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
SRC = '/content/drive/MyDrive/lyric_video_scenes'
OUT = '/content/drive/MyDrive/lyric_video_clips'
os.makedirs(OUT, exist_ok=True)
print(sorted(os.listdir(SRC)))

In [ ]:
import torch
from diffusers import LTXImageToVideoPipeline
from diffusers.utils import export_to_video
from PIL import Image

assert torch.cuda.is_available(), 'GPU on nahi hai: Runtime > Change runtime type > T4 GPU'
pipe = LTXImageToVideoPipeline.from_pretrained('Lightricks/LTX-Video', torch_dtype=torch.bfloat16)
pipe.enable_model_cpu_offload()
pipe.vae.enable_tiling()

In [ ]:
W, H, FRAMES, FPS, STEPS = 480, 832, 97, 24, 30   # memory kam pade to FRAMES = 49 kar do

STYLE = 'Flat 2D cartoon illustration with clean line art, the art style stays exactly the same. '
NEG = 'worst quality, blurry, distorted face, extra fingers, morphing, flicker, text, watermark, 3d render, photo'
SCENES = {
 '01': 'A young man and a young woman stand facing each other on a quiet street at night under a glowing street lamp. The woman slowly raises her head and looks at him with a shy smile, her dupatta sways gently in the breeze, his hair moves slightly, the lamp light flickers softly. Slow camera push in.',
 '02': 'A couple stands under a black umbrella in monsoon rain. Rain falls steadily, puddles ripple, the woman reaches her hand out to feel the rain and smiles, the man looks at her, their clothes and hair move in the wind. Slow camera pull back.',
 '03': 'Close-up of a young woman with a dupatta over her head. She blinks, lowers the dupatta slightly from her chin and parts her lips as if about to speak, her earring swings, loose hair moves in the breeze. Slow camera push in.',
 '04': 'A young man with wavy hair looks down and away, he blinks slowly and lowers his head with regret, his hair moves in the wind, the blurred dupatta in the foreground sways. Slow camera pan left.',
 '05': 'A young woman walks slowly toward the camera holding a small brass cup in both hands, she blinks and lifts the cup slightly, her dupatta and the trees sway in the warm breeze. Slow camera push in.',
 '06': 'A young man sits alone on stone steps with his hand pressed over his heart, his shoulders rise and fall as he breathes, he bows his head lower, his hair moves in the wind. Slow camera pull back.',
 '07': 'A young man laughs brightly, his shoulders shake and his head tilts back, behind him a young woman watches sadly and lowers her eyes, hair moves in the breeze. Slow camera pan right.',
 '08': 'Seen from behind, a young man stands still on an empty road while a woman in black walks away into the golden sunset haze, getting smaller, his hair and shirt move in the wind, dust drifts in the light. Slow camera pull back.',
}

def fit(img):
    s = max(W / img.width, H / img.height)
    img = img.resize((round(img.width * s), round(img.height * s)), Image.LANCZOS)
    l, t = (img.width - W) // 2, (img.height - H) // 2
    return img.crop((l, t, l + W, t + H))

for name, prompt in SCENES.items():
    out = f'{OUT}/{name}.mp4'
    if os.path.exists(out):
        print('skip', name); continue
    img = fit(Image.open(f'{SRC}/{name}.png').convert('RGB'))
    frames = pipe(image=img, prompt=STYLE + prompt, negative_prompt=NEG, width=W, height=H,
                  num_frames=FRAMES, num_inference_steps=STEPS,
                  generator=torch.Generator('cpu').manual_seed(7)).frames[0]
    export_to_video(frames, out, fps=FPS)
    print('saved', out)
    torch.cuda.empty_cache()
print('Sab ho gaya. Clips yahan hain:', OUT)